# **Day 58: Context Managers (the with Statement)**

We have used `with open(...) as f:` many times. It automatically closes the file, even if an error happens. The object that makes this possible is called a **context manager**.

**Real-life example:** A **hotel room key card**. When you enter, the lights turn on (setup). When you leave and take out the card, the lights turn off automatically (cleanup). You cannot forget to switch them off.

A context manager has two magic methods:
- `__enter__()`: runs at the **start** of the `with` block (setup)
- `__exit__()`: runs at the **end**, even if there was an error (cleanup)

In [1]:
class Demo:
    def __enter__(self):
        print("Enter: setup")
        return "value given to 'as'"

    def __exit__(self, exc_type, exc_value, traceback):
        print("Exit: cleanup, error was:", exc_type)
        return False       # False means: do not hide the error

with Demo() as v:
    print("Inside the block:", v)

Enter: setup
Inside the block: value given to 'as'
Exit: cleanup, error was: None


In [2]:
try:
    with Demo():
        print("Inside, about to fail")
        1 / 0
except ZeroDivisionError:
    print("The error was still raised after the cleanup")

Enter: setup
Inside, about to fail
Exit: cleanup, error was: <class 'ZeroDivisionError'>
The error was still raised after the cleanup


Even with the error, `__exit__` ran first. That is the whole point of a context manager.

## **A Timer Context Manager**

In [3]:
import time

class Timer:
    def __init__(self, label):
        self.label = label

    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, *args):
        self.elapsed = time.perf_counter() - self.start
        print(f"[{self.label}] {self.elapsed:.4f} seconds")

with Timer("Making a list"):
    data = [i ** 2 for i in range(500_000)]

with Timer("Sorting") as t:
    sorted(data, reverse=True)

[Making a list] 0.0225 seconds
[Sorting] 0.0061 seconds


## **The Easy Way: @contextmanager**
Instead of a class, we can write a **generator function** with one `yield`:
- Code **before** `yield` = setup
- Code **after** `yield` = cleanup (put it in `finally` so it always runs)

In [4]:
from contextlib import contextmanager

@contextmanager
def tag(name):
    print(f"<{name}>")
    try:
        yield
    finally:
        print(f"</{name}>")

with tag("html"):
    with tag("body"):
        print("  Kolkata")

<html>
<body>
  Kolkata
</body>
</html>


## **Example from GIS: Working Inside a Project Folder**
In GIS work, we often go into a project folder, create output files, and then come back. If an error happens in between, we might get stuck in the wrong folder. A context manager makes sure we **always come back**.

In [5]:
import os

@contextmanager
def working_folder(path):
    old = os.getcwd()
    os.makedirs(path, exist_ok=True)
    os.chdir(path)
    try:
        yield path
    finally:
        os.chdir(old)          # Always go back, even after an error

with working_folder("ndvi_outputs"):
    with open("kolkata_ndvi_stats.txt", "w") as f:
        f.write("mean=0.42\nmax=0.87\n")
    print("Inside  :", os.listdir("."))

print("Outside : ndvi_outputs contains", os.listdir("ndvi_outputs"))

Inside  : ['kolkata_ndvi_stats.txt']
Outside : ndvi_outputs contains ['kolkata_ndvi_stats.txt']


## **Example: Undo Changes if Something Fails (Transaction)**
If one step fails in the middle, all changes are cancelled. This is how bank transfers work.

In [6]:
@contextmanager
def transaction(account):
    backup = dict(account)
    try:
        yield account
        print("  Saved (commit)")
    except Exception as e:
        account.clear()
        account.update(backup)
        print(f"  Cancelled (rollback) because: {e}")

acc = {"Rahul": 5000, "Sima": 2000}

with transaction(acc) as a:
    a["Rahul"] -= 1000
    a["Sima"] += 1000
print(acc)

with transaction(acc) as a:
    a["Rahul"] -= 3000
    raise RuntimeError("Network failed before crediting Sima")
print(acc)

  Saved (commit)
{'Rahul': 4000, 'Sima': 3000}
  Cancelled (rollback) because: Network failed before crediting Sima
{'Rahul': 4000, 'Sima': 3000}


In the second transfer, Rahul's money was taken but the network failed. The context manager put everything back, so no money was lost.

## **suppress: Ignore a Specific Error**

In [7]:
from contextlib import suppress

with suppress(FileNotFoundError):
    os.remove("file_that_does_not_exist.txt")
print("No crash")

No crash


## **Many Context Managers in One Line**

In [8]:
with open("a.txt", "w") as fa, open("b.txt", "w") as fb:
    fa.write("Howrah")
    fb.write("Sealdah")

print(open("a.txt").read(), open("b.txt").read())

Howrah Sealdah


In [9]:
# Quick question
# In what order will the lines be printed?
# from contextlib import contextmanager
# @contextmanager
# def cm():
#     print("1"); yield; print("3")
# with cm():
#     print("2")